In [1]:
%pip install lightgbm xgboost scikit-learn numpy pandas


[notice] A new release of pip is available: 26.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Data Load


In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

DATA = '.'

train_labels = pd.read_csv(f'{DATA}/train-label.csv')
test_labels  = pd.read_csv(f'{DATA}/test-label.csv')

trainbvp   = pd.read_csv(f'{DATA}/train-bvp.csv')
traineda   = pd.read_csv(f'{DATA}/train-eda.csv')
traintemp  = pd.read_csv(f'{DATA}/train-temp.csv')
trainhr    = pd.read_csv(f'{DATA}/train-hr.csv')
trainibi   = pd.read_csv(f'{DATA}/train-ibi.csv')
trainbrain = pd.read_csv(f'{DATA}/train-brain.csv')
trainacc   = pd.read_csv(f'{DATA}/train-acc.csv')

testbvp    = pd.read_csv(f'{DATA}/test-bvp.csv')
testeda    = pd.read_csv(f'{DATA}/test-eda.csv')
testtemp   = pd.read_csv(f'{DATA}/test-temp.csv')
testhr     = pd.read_csv(f'{DATA}/test-hr.csv')
testibi    = pd.read_csv(f'{DATA}/test-ibi.csv')
testbrain  = pd.read_csv(f'{DATA}/test-brain.csv')
testacc    = pd.read_csv(f'{DATA}/test-acc.csv')

# Template required variables
TRAIN_LABEL = train_labels
TEST_DATA   = test_labels

for df in [train_labels, test_labels,
           trainbvp, traineda, traintemp, trainhr, trainibi, trainbrain, trainacc,
           testbvp,  testeda,  testtemp,  testhr,  testibi,  testbrain,  testacc]:
    df['timestamp'] = pd.to_numeric(df['timestamp'])

# ACC magnitude
for df in [trainacc, testacc]:
    df['magnitude'] = np.sqrt(df['x']**2 + df['y']**2 + df['z']**2)

# EEG log1p (middleGamma has max=4M, must transform before anything)
EEG_COLS = ['delta','theta','lowAlpha','highAlpha',
            'lowBeta','highBeta','lowGamma','middleGamma']
for df in [trainbrain, testbrain]:
    for col in EEG_COLS:
        df[col] = np.log1p(df[col])

print('Data loaded.')
print('Train labels:', train_labels.shape, '| Test labels:', test_labels.shape)
print('Train PIDs:', sorted(train_labels.pid.unique()))
print('Test  PIDs:', sorted(test_labels.pid.unique()))

Data loaded.
Train labels: (1456, 4) | Test labels: (1496, 4)
Train PIDs: ['01Z2', '70N8', '7PF3', 'CQ2G', 'D1XP', 'DT5C', 'F1ZM', 'LIUY', 'SE4Q', 'TPQI', 'Y21H']
Test  PIDs: ['13P2', '2XO3', '43JW', 'C8Q6', 'HDS9', 'NQRB', 'OL6N', 'P4DZ', 'QEYR', 'SNG7', 'TF0Y', 'WZDL']


## 1. Session Baseline Computation

In [3]:
def compute_subject_baseline(sensor_df, val_col):
    baseline = {}
    for pid, grp in sensor_df.groupby('pid'):
        vals = grp[val_col].dropna()
        baseline[pid] = (vals.mean(), vals.std() + 1e-8)
    return baseline

bl_tr_hr   = compute_subject_baseline(trainhr,   'value')
bl_tr_eda  = compute_subject_baseline(traineda,  'value')
bl_tr_temp = compute_subject_baseline(traintemp, 'value')
bl_tr_ibi  = compute_subject_baseline(trainibi,  'value')
bl_tr_acc  = compute_subject_baseline(trainacc,  'magnitude')
bl_tr_bvp  = compute_subject_baseline(trainbvp,  'value')

bl_te_hr   = compute_subject_baseline(testhr,   'value')
bl_te_eda  = compute_subject_baseline(testeda,  'value')
bl_te_temp = compute_subject_baseline(testtemp, 'value')
bl_te_ibi  = compute_subject_baseline(testibi,  'value')
bl_te_acc  = compute_subject_baseline(testacc,  'magnitude')
bl_te_bvp  = compute_subject_baseline(testbvp,  'value')

print('Baselines computed for all sensors including BVP.')

Baselines computed for all sensors including BVP.


## 2. Feature Extraction — Multi-Window + BVP

In [4]:
WINDOWS_MS = [2500, 5000, 10000]


def win_stats(vals, prefix):
    """Statistical features from a 1D array."""
    feat = {}
    n = len(vals)
    if n >= 2:
        feat[f'{prefix}_mean']  = np.mean(vals)
        feat[f'{prefix}_std']   = np.std(vals)
        feat[f'{prefix}_range'] = np.max(vals) - np.min(vals)
        feat[f'{prefix}_slope'] = np.polyfit(np.arange(n), vals, 1)[0]
        feat[f'{prefix}_p25']   = np.percentile(vals, 25)
        feat[f'{prefix}_p75']   = np.percentile(vals, 75)
    else:
        for s in ['mean','std','range','slope','p25','p75']:
            feat[f'{prefix}_{s}'] = np.nan
    return feat


def extract_all_features(label_df, is_train,
                         hr_df, eda_df, temp_df, ibi_df, acc_df, brain_df, bvp_df,
                         bl_hr, bl_eda, bl_temp, bl_ibi, bl_acc, bl_bvp):

    for df in [hr_df, eda_df, temp_df, ibi_df, acc_df, brain_df, bvp_df]:
        df.sort_values(['pid','timestamp'], inplace=True)

    records = []
    for _, row in label_df.iterrows():
        pid, ts = row['pid'], row['timestamp']
        feat = {'pid': pid, 'timestamp': ts}
        if is_train:
            feat['arousal'] = row['arousal']

        # Session baselines as direct features (model can use absolute physiological level)
        feat['bl_hr']   = bl_hr.get(pid,   (np.nan,1))[0]
        feat['bl_eda']  = bl_eda.get(pid,  (np.nan,1))[0]
        feat['bl_temp'] = bl_temp.get(pid, (np.nan,1))[0]
        feat['bl_ibi']  = bl_ibi.get(pid,  (np.nan,1))[0]
        feat['bl_acc']  = bl_acc.get(pid,  (np.nan,1))[0]
        feat['bl_bvp']  = bl_bvp.get(pid,  (np.nan,1))[0]

        for hw in WINDOWS_MS:
            wl = f'w{hw//1000}s'

            def get_win(df_, col):
                s = df_[df_.pid == pid]
                return s[(s.timestamp >= ts - hw) & (s.timestamp < ts + hw)][col].values

            # ── HR ────────────────────────────────────────────────
            hr_v = get_win(hr_df, 'value')
            feat.update(win_stats(hr_v, f'hr_{wl}'))
            bl_m, bl_s = bl_hr.get(pid, (np.nan, 1))
            feat[f'hr_{wl}_dev'] = (np.mean(hr_v) - bl_m) / bl_s if len(hr_v) >= 1 else np.nan

            # ── EDA ───────────────────────────────────────────────
            eda_v = get_win(eda_df, 'value')
            feat.update(win_stats(eda_v, f'eda_{wl}'))
            bl_m, bl_s = bl_eda.get(pid, (np.nan, 1))
            if len(eda_v) >= 1:
                zr = np.mean(eda_v == 0)
                feat[f'eda_{wl}_zero_ratio'] = zr
                feat[f'eda_{wl}_valid']       = 0 if zr > 0.5 else 1
                feat[f'eda_{wl}_dev']         = (np.mean(eda_v) - bl_m) / bl_s
                # Non-zero EDA mean (ignores artifact zeros)
                nz = eda_v[eda_v != 0]
                feat[f'eda_{wl}_nz_mean'] = np.mean(nz) if len(nz) >= 1 else np.nan
            else:
                feat[f'eda_{wl}_zero_ratio'] = np.nan
                feat[f'eda_{wl}_valid']       = 0
                feat[f'eda_{wl}_dev']         = np.nan
                feat[f'eda_{wl}_nz_mean']     = np.nan

            # ── TEMP ──────────────────────────────────────────────
            temp_v = get_win(temp_df, 'value')
            feat.update(win_stats(temp_v, f'temp_{wl}'))
            bl_m, bl_s = bl_temp.get(pid, (np.nan, 1))
            feat[f'temp_{wl}_dev'] = (np.mean(temp_v) - bl_m) / bl_s if len(temp_v) >= 1 else np.nan

            # ── IBI / HRV ─────────────────────────────────────────
            ibi_v = get_win(ibi_df, 'value')
            bl_m, bl_s = bl_ibi.get(pid, (np.nan, 1))
            if len(ibi_v) >= 2:
                feat[f'ibi_{wl}_mean']  = np.mean(ibi_v)
                feat[f'ibi_{wl}_std']   = np.std(ibi_v)
                feat[f'ibi_{wl}_range'] = np.max(ibi_v) - np.min(ibi_v)
                diffs = np.diff(ibi_v)
                feat[f'ibi_{wl}_rmssd'] = np.sqrt(np.mean(diffs**2))
                # pNN50: proportion of successive differences > 50ms
                feat[f'ibi_{wl}_pnn50'] = np.mean(np.abs(diffs) > 50)
                feat[f'ibi_{wl}_dev']   = (np.mean(ibi_v) - bl_m) / bl_s
            else:
                for s in ['mean','std','range','rmssd','pnn50','dev']:
                    feat[f'ibi_{wl}_{s}'] = np.nan

            # ── ACC ───────────────────────────────────────────────
            acc_v = get_win(acc_df, 'magnitude')
            bl_m, bl_s = bl_acc.get(pid, (np.nan, 1))
            if len(acc_v) >= 5:
                feat[f'acc_{wl}_mean']   = np.mean(acc_v)
                feat[f'acc_{wl}_std']    = np.std(acc_v)
                feat[f'acc_{wl}_energy'] = np.mean(acc_v**2)
                feat[f'acc_{wl}_dev']    = (np.mean(acc_v) - bl_m) / bl_s
            else:
                for s in ['mean','std','energy','dev']:
                    feat[f'acc_{wl}_{s}'] = np.nan

            # ── BVP ───────────────────────────────────────────────
            # BVP at 64Hz: use std (pulse amplitude variability) and range
            bvp_v = get_win(bvp_df, 'value')
            bl_m, bl_s = bl_bvp.get(pid, (np.nan, 1))
            if len(bvp_v) >= 10:
                feat[f'bvp_{wl}_std']   = np.std(bvp_v)
                feat[f'bvp_{wl}_range'] = np.max(bvp_v) - np.min(bvp_v)
                feat[f'bvp_{wl}_dev']   = (np.mean(bvp_v) - bl_m) / bl_s
                # Approximate pulse amplitude: IQR of BVP
                feat[f'bvp_{wl}_iqr']   = np.percentile(bvp_v,75) - np.percentile(bvp_v,25)
            else:
                for s in ['std','range','dev','iqr']:
                    feat[f'bvp_{wl}_{s}'] = np.nan

            # ── EEG ───────────────────────────────────────────────
            s_eeg = brain_df[brain_df.pid == pid]
            win_eeg = s_eeg[(s_eeg.timestamp >= ts - hw) & (s_eeg.timestamp < ts + hw)]
            eps = 1e-8
            if len(win_eeg) >= 1:
                for col in EEG_COLS:
                    feat[f'eeg_{col}_{wl}'] = win_eeg[col].mean()
                th = feat[f'eeg_theta_{wl}']
                la = feat[f'eeg_lowAlpha_{wl}']
                ha = feat[f'eeg_highAlpha_{wl}']
                lb = feat[f'eeg_lowBeta_{wl}']
                hb = feat[f'eeg_highBeta_{wl}']
                lg = feat[f'eeg_lowGamma_{wl}']
                de = feat[f'eeg_delta_{wl}']
                feat[f'eeg_theta_alpha_{wl}']  = th / (la + ha + eps)
                feat[f'eeg_beta_alpha_{wl}']   = (lb + hb) / (la + ha + eps)
                feat[f'eeg_hbeta_lgamma_{wl}'] = hb / (lg + eps)
                feat[f'eeg_engage_{wl}']        = hb / (de + th + eps)  # engagement index
            else:
                for col in EEG_COLS:
                    feat[f'eeg_{col}_{wl}'] = np.nan
                for r in ['theta_alpha','beta_alpha','hbeta_lgamma','engage']:
                    feat[f'eeg_{r}_{wl}'] = np.nan

        records.append(feat)

    return pd.DataFrame(records)


print('Feature extraction function ready.')

Feature extraction function ready.


In [5]:
print('Extracting TRAIN features (may take 4–6 min)...')
train_feats = extract_all_features(
    train_labels, True,
    trainhr, traineda, traintemp, trainibi, trainacc, trainbrain, trainbvp,
    bl_tr_hr, bl_tr_eda, bl_tr_temp, bl_tr_ibi, bl_tr_acc, bl_tr_bvp
)
train_feats.insert(0, 'id', train_labels['id'].values)
print('Train features:', train_feats.shape)

Extracting TRAIN features (may take 4–6 min)...
Train features: (1456, 160)


In [6]:
print('Extracting TEST features...')
test_feats = extract_all_features(
    test_labels, False,
    testhr, testeda, testtemp, testibi, testacc, testbrain, testbvp,
    bl_te_hr, bl_te_eda, bl_te_temp, bl_te_ibi, bl_te_acc, bl_te_bvp
)
test_feats.insert(0, 'id', test_labels['id'].values)
print('Test features:', test_feats.shape)

Extracting TEST features...
Test features: (1496, 159)


## 3. Lag Features (temporal context within subject)

In [7]:
# Key signals to lag: the most arousal-correlated features from v2
LAG_KEYS = ['hr_w5s_mean','hr_w5s_dev','eda_w5s_mean','eda_w5s_dev',
            'temp_w5s_mean','temp_w5s_dev','eda_w5s_zero_ratio',
            'ibi_w5s_rmssd','bvp_w5s_std']
LAG_COLS = [c for c in LAG_KEYS if c in train_feats.columns]


def add_lag_features(df, cols, lags=(1, 2)):
    df = df.sort_values(['pid','timestamp']).copy()
    for lag in lags:
        for col in cols:
            df[f'{col}_lag{lag}'] = df.groupby('pid')[col].shift(lag)
    # Rolling mean of last 3 windows (running physiological trend)
    for col in cols:
        df[f'{col}_roll3'] = df.groupby('pid')[col].transform(
            lambda x: x.shift(1).rolling(3, min_periods=1).mean()
        )
    return df


train_feats = add_lag_features(train_feats, LAG_COLS)
test_feats  = add_lag_features(test_feats,  LAG_COLS)

META_COLS = ['id','pid','timestamp','arousal']
FEAT_COLS = [c for c in train_feats.columns if c not in META_COLS]

print(f'Total features: {len(FEAT_COLS)}')
nan_summary = train_feats[FEAT_COLS].isnull().sum()
high_nan = nan_summary[nan_summary > 200]
print(f'Features with >200 NaN ({len(high_nan)} features):')
print(high_nan.sort_values(ascending=False).head(10))

Total features: 183
Features with >200 NaN (24 features):
ibi_w2s_std           812
ibi_w2s_range         812
ibi_w2s_rmssd         812
ibi_w2s_pnn50         812
ibi_w2s_dev           812
ibi_w2s_mean          812
ibi_w5s_rmssd_lag2    637
ibi_w5s_rmssd_lag1    634
ibi_w5s_pnn50         631
ibi_w5s_mean          631
dtype: int64


## 4. Manual Class Weights — Amplify Classes 1 and 5

In [8]:
# Compute class frequencies
train_feats_sorted = train_feats.sort_values(['pid','timestamp']).reset_index(drop=True)
X_all  = train_feats_sorted[FEAT_COLS].values.astype(np.float32)
y_all  = (train_feats_sorted['arousal'].values - 1).astype(int)  # 0-indexed
pids   = train_feats_sorted['pid'].values
X_test = test_feats[FEAT_COLS].values.astype(np.float32)

counts = np.bincount(y_all, minlength=5)
print('Class counts (0=Arousal1 ... 4=Arousal5):', counts)

# Manual weights: base = inverse frequency, then BOOST rare classes further
# Arousal 1 (idx 0) and Arousal 5 (idx 4) get extra 3x boost on top of inverse freq
base_w   = len(y_all) / (5 * counts.astype(float))
boost    = np.array([3.0, 1.0, 1.0, 1.0, 3.0])   # amplify classes 1 and 5
CLASS_W  = base_w * boost
CLASS_W  = CLASS_W / CLASS_W.mean()              # normalise so mean=1

print('Final class weights:', {f'Arousal {i+1}': round(w,2) for i, w in enumerate(CLASS_W)})

sample_weights = CLASS_W[y_all]

Class counts (0=Arousal1 ... 4=Arousal5): [ 55 430 554 345  72]
Final class weights: {'Arousal 1': np.float64(2.64), 'Arousal 2': np.float64(0.11), 'Arousal 3': np.float64(0.09), 'Arousal 4': np.float64(0.14), 'Arousal 5': np.float64(2.02)}


## 5. LOSO CV — LightGBM + XGBoost with Boosted Class Weights

In [9]:
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import balanced_accuracy_score, classification_report

TRAIN_PIDS = sorted(train_feats_sorted['pid'].unique())
SEEDS_LGB  = [42, 7, 123, 13, 99]
SEEDS_XGB  = [42, 7, 123]

oof_lgb  = np.zeros((len(train_feats_sorted), 5), dtype=np.float64)
oof_xgb  = np.zeros((len(train_feats_sorted), 5), dtype=np.float64)
test_lgb = np.zeros((len(test_feats), 5), dtype=np.float64)
test_xgb = np.zeros((len(test_feats), 5), dtype=np.float64)

loso_lgb_scores, loso_xgb_scores = [], []


def get_lgb_params(seed):
    return dict(
        objective='multiclass', num_class=5, metric='multi_logloss',
        num_leaves=31, learning_rate=0.03,
        feature_fraction=0.7, bagging_fraction=0.8, bagging_freq=5,
        min_child_samples=10,   # lower → model can fit rare classes with few samples
        lambda_l1=0.3, lambda_l2=0.3,
        max_depth=6, verbose=-1, seed=seed, n_jobs=-1
    )


def get_xgb_params(seed):
    return dict(
        objective='multi:softprob', num_class=5, eval_metric='mlogloss',
        max_depth=4, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.7,
        min_child_weight=5,    # lower → can fit rare classes
        reg_alpha=0.3, reg_lambda=0.3,
        seed=seed, verbosity=0, nthread=-1
    )


for fold_pid in TRAIN_PIDS:
    tr_mask = pids != fold_pid
    va_mask = pids == fold_pid

    X_tr, y_tr = X_all[tr_mask], y_all[tr_mask]
    X_va, y_va = X_all[va_mask], y_all[va_mask]
    sw_tr = sample_weights[tr_mask]

    fold_oof_lgb  = np.zeros((va_mask.sum(), 5))
    fold_oof_xgb  = np.zeros((va_mask.sum(), 5))
    fold_test_lgb = np.zeros((len(test_feats), 5))
    fold_test_xgb = np.zeros((len(test_feats), 5))

    # ── LightGBM multi-seed ────────────────────────────────────────────
    for seed in SEEDS_LGB:
        dtr = lgb.Dataset(X_tr, label=y_tr, weight=sw_tr)
        dva = lgb.Dataset(X_va, label=y_va, reference=dtr)
        m = lgb.train(
            get_lgb_params(seed), dtr,
            num_boost_round=1200,
            valid_sets=[dva],
            callbacks=[lgb.early_stopping(80, verbose=False),
                       lgb.log_evaluation(period=-1)],
        )
        fold_oof_lgb  += m.predict(X_va)    / len(SEEDS_LGB)
        fold_test_lgb += m.predict(X_test)  / len(SEEDS_LGB)

    # ── XGBoost multi-seed ────────────────────────────────────────────
    for seed in SEEDS_XGB:
        dtr_x = xgb.DMatrix(X_tr, label=y_tr, weight=sw_tr)
        dva_x = xgb.DMatrix(X_va, label=y_va)
        m_x = xgb.train(
            get_xgb_params(seed), dtr_x,
            num_boost_round=1200,
            evals=[(dva_x, 'val')],
            early_stopping_rounds=80,
            verbose_eval=False,
        )
        p = m_x.predict(dva_x).reshape(-1, 5)
        fold_oof_xgb  += p                                      / len(SEEDS_XGB)
        fold_test_xgb += m_x.predict(xgb.DMatrix(X_test)).reshape(-1, 5) / len(SEEDS_XGB)

    oof_lgb[va_mask]  = fold_oof_lgb
    oof_xgb[va_mask]  = fold_oof_xgb
    test_lgb         += fold_test_lgb / len(TRAIN_PIDS)
    test_xgb         += fold_test_xgb / len(TRAIN_PIDS)

    ba_lgb = balanced_accuracy_score(y_va, fold_oof_lgb.argmax(axis=1))
    ba_xgb = balanced_accuracy_score(y_va, fold_oof_xgb.argmax(axis=1))
    loso_lgb_scores.append(ba_lgb)
    loso_xgb_scores.append(ba_xgb)
    print(f'  {fold_pid} — LGB: {ba_lgb:.4f} | XGB: {ba_xgb:.4f}')

print(f'\nLOSO LGB mean: {np.mean(loso_lgb_scores):.4f} ± {np.std(loso_lgb_scores):.4f}')
print(f'LOSO XGB mean: {np.mean(loso_xgb_scores):.4f} ± {np.std(loso_xgb_scores):.4f}')

  01Z2 — LGB: 0.4944 | XGB: 0.3556
  70N8 — LGB: 0.1265 | XGB: 0.1364
  7PF3 — LGB: 0.1588 | XGB: 0.2290
  CQ2G — LGB: 0.0000 | XGB: 0.1819
  D1XP — LGB: 0.2745 | XGB: 0.2524
  DT5C — LGB: 0.2355 | XGB: 0.1473
  F1ZM — LGB: 0.0830 | XGB: 0.1163
  LIUY — LGB: 0.4580 | XGB: 0.2316
  SE4Q — LGB: 0.2921 | XGB: 0.4737
  TPQI — LGB: 0.1188 | XGB: 0.0621
  Y21H — LGB: 0.1426 | XGB: 0.1457

LOSO LGB mean: 0.2167 ± 0.1465
LOSO XGB mean: 0.2120 ± 0.1121


## 6. OOF Blend Optimisation + Threshold Shifting

In [10]:
# Step 6a: find best LGB:XGB blend weight
best_ba_blend, best_w = 0, 0.5
for w in np.arange(0.0, 1.01, 0.05):
    blend = w * oof_lgb + (1-w) * oof_xgb
    ba    = balanced_accuracy_score(y_all, blend.argmax(axis=1))
    if ba > best_ba_blend:
        best_ba_blend, best_w = ba, w

print(f'Best blend weight LGB={best_w:.2f} → OOF BA (argmax): {best_ba_blend:.4f}')
oof_blend  = best_w * oof_lgb  + (1 - best_w) * oof_xgb
test_blend = best_w * test_lgb + (1 - best_w) * test_xgb

# Step 6b: threshold optimisation per class
# BA = mean per-class recall. We can shift class probabilities to force
# the model to predict rare classes more often.
# Method: multiply each class's probability column by a threshold factor,
# then argmax. Optimise factors on OOF to maximise BA.

best_ba_thresh = best_ba_blend
best_factors   = np.ones(5)

# Search over scale factors for classes 1 and 5 (indices 0 and 4)
# Keep factors for 2,3,4 (indices 1,2,3) at 1.0
for f1 in np.arange(1.0, 5.1, 0.5):     # scale factor for Arousal 1
    for f5 in np.arange(1.0, 5.1, 0.5): # scale factor for Arousal 5
        factors = np.array([f1, 1.0, 1.0, 1.0, f5])
        scaled  = oof_blend * factors
        pred    = scaled.argmax(axis=1)
        ba      = balanced_accuracy_score(y_all, pred)
        if ba > best_ba_thresh:
            best_ba_thresh = ba
            best_factors   = factors.copy()

print(f'Best threshold factors: {best_factors}')
print(f'OOF BA after threshold optimisation: {best_ba_thresh:.4f}')

# Apply best factors to final predictions
test_pred_raw    = test_blend.argmax(axis=1) + 1
test_pred_thresh = (test_blend * best_factors).argmax(axis=1) + 1

print('\nTest distribution (argmax):', dict(pd.Series(test_pred_raw).value_counts().sort_index()))
print('Test distribution (thresh): ', dict(pd.Series(test_pred_thresh).value_counts().sort_index()))

Best blend weight LGB=1.00 → OOF BA (argmax): 0.2016
Best threshold factors: [1.  1.  1.  1.  3.5]
OOF BA after threshold optimisation: 0.3071

Test distribution (argmax): {1: np.int64(252), 2: np.int64(198), 3: np.int64(561), 4: np.int64(169), 5: np.int64(316)}
Test distribution (thresh):  {1: np.int64(167), 2: np.int64(112), 3: np.int64(299), 4: np.int64(76), 5: np.int64(842)}


## 7. Final Diagnosis on OOF

In [11]:
final_oof_pred = (oof_blend * best_factors).argmax(axis=1)
print('=== OOF Classification Report (with threshold) ===')
print(classification_report(y_all, final_oof_pred,
                            target_names=[f'Arousal {i+1}' for i in range(5)]))

print('Per-subject LOSO BA (LGB):')
for pid, ba in zip(TRAIN_PIDS, loso_lgb_scores):
    flag = ' ← LOW' if ba < 0.25 else ''
    print(f'  {pid}: {ba:.4f}{flag}')

=== OOF Classification Report (with threshold) ===
              precision    recall  f1-score   support

   Arousal 1       0.01      0.02      0.01        55
   Arousal 2       0.34      0.25      0.29       430
   Arousal 3       0.36      0.16      0.22       554
   Arousal 4       0.34      0.21      0.26       345
   Arousal 5       0.11      0.90      0.20        72

    accuracy                           0.23      1456
   macro avg       0.23      0.31      0.19      1456
weighted avg       0.32      0.23      0.24      1456

Per-subject LOSO BA (LGB):
  01Z2: 0.4944
  70N8: 0.1265 ← LOW
  7PF3: 0.1588 ← LOW
  CQ2G: 0.0000 ← LOW
  D1XP: 0.2745
  DT5C: 0.2355 ← LOW
  F1ZM: 0.0830 ← LOW
  LIUY: 0.4580
  SE4Q: 0.2921
  TPQI: 0.1188 ← LOW
  Y21H: 0.1426 ← LOW


# Generating Final Submissions


In [12]:
# Use threshold-optimised predictions (better BA on OOF)
submission = pd.DataFrame({
    'id':      test_feats['id'].values,
    'arousal': test_pred_thresh,
})

submission.to_csv('submission_v14.csv', index=False)
print('submission.csv saved.')
print(f'Shape: {submission.shape}')
print(submission['arousal'].value_counts().sort_index())

submission.csv saved.
Shape: (1496, 2)
arousal
1    167
2    112
3    299
4     76
5    842
Name: count, dtype: int64
